In [2]:
import torch
from deepmd.pt.model.descriptor.se_a import DescrptSeA
from deepmd.pt.model.task.ener import EnergyFittingNet
from deepmd.pt.utils.nlist import extend_input_and_build_neighbor_list

# 1. 设置设备
device = torch.device('cpu')

# 2. 设置参数
type_map = ["O", "H"]
rcut = 6.0
sel = [40, 40]
rcut_smth = 0.5
descriptor = DescrptSeA(
    rcut=rcut,
    rcut_smth=rcut_smth,
    sel=sel,
    neuron=[8, 8],
    axis_neuron=8,
    precision="float64",
).to(device)

# 4. 准备输入数据
coord = torch.tensor([
    [0.0, 0.0, 0.0],
    [0.76, 0.59, 0.0],
    [-0.76, 0.59, 0.0],
    [3.0, 0.0, 0.0],
    [3.76, 0.59, 0.0],
    [2.24, 0.59, 0.0],
], dtype=torch.float64, device=device)

atype = torch.tensor([0, 1, 1, 0, 1, 1], dtype=torch.int32, device=device)

cell = torch.diag(torch.tensor([10.0, 10.0, 10.0], dtype=torch.float64, device=device))

# 5. 添加批次维度
coord = coord.unsqueeze(0)  # [1, 6, 3]
atype = atype.unsqueeze(0)  # [1, 6]
cell = cell.unsqueeze(0)    # [1, 3, 3]

# 6. 构建邻居列表
extended_coord, extended_atype, mapping, nlist = extend_input_and_build_neighbor_list(
    coord, atype, rcut, sel, box=cell
)

# 7. 计算描述符
desc = descriptor(extended_coord, extended_atype, nlist)

# 8. 输出结果
print(f"描述符向量维度: {descriptor.get_dim_out()}")
print(f"输出形状: {desc[0].shape}")
#print(desc)

To get the best performance, it is recommended to adjust the number of threads by setting the environment variables OMP_NUM_THREADS, DP_INTRA_OP_PARALLELISM_THREADS, and DP_INTER_OP_PARALLELISM_THREADS. See https://deepmd.rtfd.io/parallelism/ for more information.


描述符向量维度: 64
输出形状: torch.Size([1, 6, 64])


In [3]:
desc[0].shape

torch.Size([1, 6, 64])

In [4]:
fitting_net = EnergyFittingNet(
    n_out=1,
    dim_descrpt=descriptor.get_dim_out(),
    n_hidden=[32, 16],
    ntypes=len(type_map),   # 这里是 2
    type_map=type_map,
    activation_function="tanh",
    precision="float64",
).to(device)

In [5]:
fitting_net(desc[0], atype)

{'energy': tensor([[[-0.4331],
          [-0.4332],
          [-0.4332],
          [-0.4331],
          [-0.4332],
          [-0.4332]]], dtype=torch.float64, grad_fn=<WhereBackward0>)}

In [6]:
import torch
import yaml
from deepmd.pt.model.descriptor.se_a import DescrptSeA
from deepmd.pt.model.task.ener import EnergyFittingNet
from les import Les  # 请根据你的实际路径调整
from hybridles import HybridLESAtomicModel

In [7]:
device = torch.device('cpu')
type_map = ["O", "H"]
rcut = 6.0
rcut_smth = 0.5
sel = [40, 40]

# 描述符（轻量化：输出 8x8 = 64 维）
descriptor = DescrptSeA(
    rcut=rcut,
    rcut_smth=rcut_smth,
    sel=sel,
    neuron=[8, 8],
    axis_neuron=8,
    precision="float64",
).to(device)

# 短程拟合网络（输入=64维，输出=1）
fitting_net = EnergyFittingNet(
    ntypes=2,
    #n_out=1,
    dim_descrpt=descriptor.get_dim_out(),
    n_hidden=[32, 16],
    activation_function="tanh",
    precision="float64",
).to(device)

# LES 模型
with open('example/input.yaml', 'r') as f:
    les_config = yaml.safe_load(f)
les_config['use_atomwise'] = True  # 让 LES 内部通过 desc 生成 q
les_model = Les(les_arguments=les_config).to(device)

# ---------- 2. 构建混合模型 ----------
model = HybridLESAtomicModel(
    descriptor=descriptor,
    fitting_net=fitting_net,
    les_model=les_model,
    type_map=type_map,
).to(device)

# ---------- 3. 准备输入数据（两个水分子） ----------
coord = torch.tensor([
    [0.0, 0.0, 0.0],
    [0.76, 0.59, 0.0],
    [-0.76, 0.59, 0.0],
    [3.0, 0.0, 0.0],
    [3.76, 0.59, 0.0],
    [2.24, 0.59, 0.0],
], dtype=torch.float64, device=device)

atype = torch.tensor([0, 1, 1, 0, 1, 1], dtype=torch.int32, device=device)
cell = torch.diag(torch.tensor([10.0, 10.0, 10.0], dtype=torch.float64, device=device))

# 添加 batch 维度
coord = coord.unsqueeze(0)   # [1, 6, 3]
atype = atype.unsqueeze(0)   # [1, 6]
cell = cell.unsqueeze(0)     # [1, 3, 3]

In [8]:
with torch.no_grad():
    E_pred = model(coord, atype, cell)
    print(f"预测的总能量: {E_pred.item():.6f} eV")

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float